### Main paper plot 2

X-axis: time taken corresponding to the iteration (log scale)

Y-axis: F(S, Q)

Each curve represents the performance of a method.

In [ ]:
import torch
import pickle
import os
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm

In [ ]:
from plot_utils import crop_pdf_with_fitz, crop_pdf_with_pdfcrop, crop_pdf_with_pypdf,\
    legend_labels, method_label_map, methods, legend_color_map, legend_marker_map

In [ ]:
%load_ext autoreload
%autoreload 1

%aimport plot_utils

In [ ]:
# Collect data from text files for all datasets
datasets = ['msmarco', 'hotpotqa', 'fever', 'pooled', 'science', 'technology', 'writing']
time_map, max_time_vals = plot_utils.get_time_data(datasets)

In [ ]:
max_time_vals

In [ ]:
# Load score data
score_map = {ds: {} for ds in datasets}
ind_map = {ds: {} for ds in datasets}

for ds in datasets:
    for method in methods:
        if method == "gold":
            continue
        inds, scores = plot_utils.get_score_data(ds, method, k=10)
        print(f"Shapes for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}")
        assert inds.shape == scores.shape, f"Shape mismatch for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}"
        assert inds.shape[0] > 10
        score_map[ds][method] = scores.mean(dim=0).numpy()
        ind_map[ds][method] = inds

In [ ]:
def plot_paper(dataset_name, desired_methods, y_label=True, k_cutoff=4):
    plt.clf()
    fig, ax = plt.subplots(figsize=(8, 6))
    markersize = 10

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
        'lines.markersize': markersize
    })

    exact_greedy_time = np.array(time_map[dataset_name]["exact greedy"] * 10)
    print(exact_greedy_time)

    for method in desired_methods:
        query_time = time_map[dataset_name][method]
        scores = score_map[dataset_name][method].tolist()
        query_times = np.array([query_time * i for i in range(1, len(scores) + 1)])

        if "bypass" in method:
            scores = scores[:k_cutoff]
            query_times = query_times[:k_cutoff]

        efficiency = exact_greedy_time / query_times

        print(f"Method: {method}, Time: {query_time}, Score: {scores}")
        ax.plot(
            efficiency,
            scores,
            label=plot_utils.method_label_map[method],
            color=plot_utils.legend_color_map[plot_utils.method_label_map[method]],
            marker=plot_utils.legend_marker_map[plot_utils.method_label_map[method]],
            linewidth=2,
            markersize=14,
        )

    # ax.set_title(f'{dataset_name}: F(S) vs K', fontsize=20)
    # ax.set_xlabel(r'$\textbf{Retrieval time (sec)}$', fontsize=48)      # Increased axis label size
    ax.set_xlabel(r'$\textbf{Efficiency}\rightarrow$', fontsize=48)      # Increased axis label size
    if y_label:
        ax.set_ylabel(r'$\textbf{Avg}\quad\pmb{F(S_K, Q)}$', fontsize=48) # Increased axis label size

    # Xticks logscale
    ax.set_xscale('log')
    # Explicitly set tick label sizes
    ax.tick_params(axis='x', labelsize=50)  # Smaller X-axis tick labels
    ax.tick_params(axis='y', labelsize=50)  # Smaller Y-axis tick labels

    ax.set_xticks([1, 100, 10000])
    # Get current x-tick values and format them with LaTeX bold
    xticks = ax.get_xticks()
    # Display in 10^i format, and use bold for the numbers
    powers = [f"10^{int(np.log10(v))}" for v in xticks if v > 0]
    xticklabels = [fr'$\mathbf{{{p}}}$' for p in powers]
    ax.set_xticklabels(xticklabels)

    # Get current y-tick values and format them with LaTeX bold
    yticks = ax.get_yticks()
    yticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in yticks]
    ax.set_yticklabels(yticklabels)

    ax.xaxis.set_label_coords(0.45, -0.2)

    # ax.legend(fontsize=12)
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(f'./notebooks/plots/{dataset_name.lower()}_plot2.pdf')
    plt.show()

In [ ]:
def plot_legend_only(desired_methods, filename, ncol=3, auto_crop=True):
    # Create a figure for legend only
    fig, ax = plt.subplots(figsize=(8, 6))

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
    })
    
    for method in desired_methods:
        ax.plot(
            [], [],
            label=plot_utils.method_label_map[method],
            color=plot_utils.legend_color_map[plot_utils.method_label_map[method]],
            marker=plot_utils.legend_marker_map[plot_utils.method_label_map[method]],
            linewidth=2,
            markersize=10,
        )

    # Hide the plot area
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Create legend
    legend = ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=ncol,
        fontsize=25,
        frameon=False
    )

    # Save the figure
    output_path = f'./notebooks/plots/{filename}.pdf'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()
    
    # Auto-crop the PDF if requested
    if auto_crop:
        # Try methods in order of preference
        cropped_path = crop_pdf_with_pdfcrop(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_fitz(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_pypdf(output_path)
        
        if cropped_path:
            # Replace original with cropped version
            os.rename(cropped_path, output_path)
            print(f"PDF automatically cropped: {output_path}")
        else:
            print("Auto-cropping failed, using matplotlib's bbox_inches='tight' only")

In [ ]:
dms = ['disco - 1', 'submodlib ltl 0.5', 'exact greedy', 'MUVERA iid', 'submodlib lazy', 'WARP iid', 'submodlib stochastic 0.5', 'ColBERT iid']

In [ ]:
plot_legend_only(dms, "plot2_legend", ncol=4)

In [ ]:
# dms = ['submodlib ltl 0.5', 'exact greedy', 'submodlib lazy', 'submodlib stochastic 0.5']
plot_paper("msmarco", dms, y_label=True, k_cutoff=10)

In [ ]:
plot_paper("fever", dms, y_label=False, k_cutoff=10)

In [ ]:
plot_paper("hotpotqa", dms, y_label=False, k_cutoff=10)

In [ ]:
plot_paper("pooled", dms, y_label=False, k_cutoff=10)

In [ ]:
plot_paper("science", dms, y_label=True, k_cutoff=10)

In [ ]:
plot_paper("writing", dms, y_label=False, k_cutoff=10)

In [ ]:
plot_paper("technology", dms, y_label=False, k_cutoff=10)